# Oracle BTC up/down : ouverture et clôture

Backfill de 40 jours pour les marchés BTC 5 minutes et BTC 15 minutes.

L'archive qui contient le prix **qui a réglé** le marché est Gamma : `eventMetadata.priceToBeat` à l'ouverture de la fenêtre, `eventMetadata.finalPrice` à la clôture. Les deux viennent du TWAP Chainlink BTC/USD 60 secondes. Gamma n'écrit ces champs qu'après la fin de la fenêtre.

Le prix affiché en direct sur le site est un autre appel, `https://polymarket.com/api/crypto/crypto-price`. Il expose bien un `openPrice` pendant que la fenêtre est ouverte, et un `closePrice` qui bouge. Sur les fenêtres déjà réglées, ces nombres ne sont pas ceux de Gamma : l'écart se mesure plus bas.

Deux fichiers, pour ne pas mélanger un prix d'ouverture (connu dans Gamma seulement à la fin) avec le prix de clôture :

- `data/price_oracle_opening.parquet`
- `data/price_oracle_close.parquet`

`oracle_ts` est l'instant auquel l'échantillon se rapporte. `gamma_closed_time` est l'instant où Gamma a publié le résultat.

In [ ]:
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.datasets import publication_lag_summary, save_oracle_frames, split_oracle_frames
from utils.gamma import pull_windows
from utils.live import compare_with_gamma, probe_live_clock

DATA = ROOT / "data"
LOOKBACK_DAYS = 40
end_max = datetime.now(timezone.utc)
end_min = end_max - timedelta(days=LOOKBACK_DAYS)
end_min, end_max

In [ ]:
rows = pull_windows(end_min, end_max, timeframes=("5m", "15m"))
opening, close = split_oracle_frames(rows)
opening_path, close_path = save_oracle_frames(opening, close, DATA)
print(opening_path, len(opening))
print(close_path, len(close))
print("rows without an opening price", sum(r["price_to_beat"] is None for r in rows))
print("rows without a close price", sum(r["final_price"] is None for r in rows))

In [ ]:
opening.groupby("timeframe").size().rename("windows").to_frame().join(
    close.groupby("timeframe").size().rename("closes")
)

## Lag de publication Gamma

`close_to_outcome` : secondes entre la fin de la fenêtre et `closedTime`. C'est le lag utilisable. La médiane est d'environ une minute : c'est le moment où Gamma publie le résultat.

`close_to_last_gamma_write` : secondes entre la fin de la fenêtre et `updatedAt`. Ce champ est réécrit plus tard, souvent le lendemain. Il ne date pas l'arrivée de `finalPrice`.

`open_sample_to_outcome` : secondes entre le début de la fenêtre et `closedTime`. Le délai inclut toute la durée du pari. Il rappelle que, dans Gamma, le prix d'ouverture de ce fichier n'était pas connu à `oracle_ts`.

In [ ]:
publication_lag_summary(opening, close)

## Horloge du prix affiché en live

Pour la fenêtre en cours, `open_price` doit être rempli et `close_price` vide. `rtt_ms` est le temps aller-retour HTTP. `server_date_minus_local_after_ms` et `quote_ts_minus_local_after_ms` comparent l'horloge du serveur à l'horloge locale au moment où la réponse est reçue : une valeur négative veut dire que l'horodatage reçu est déjà dans le passé.

In [ ]:
live = probe_live_clock()
live[
    [
        "timeframe",
        "window_start",
        "http_status",
        "rtt_ms",
        "server_date_minus_local_after_ms",
        "quote_ts_minus_local_after_ms",
        "open_price",
        "close_price",
        "completed",
    ]
]

## Le prix du site n'est pas le prix de règlement

Même appel que la page, sur les dernières fenêtres déjà présentes dans Gamma. `open_gamma_minus_site` et `close_gamma_minus_site` sont en dollars.

In [ ]:
compare_with_gamma(opening, close, n=6)